<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2026/images/logo.png?raw=true" alt="2026年度ゲノム情報解析入門" height="100px" align="middle">

<div align="right"><a href="https://github.com/CropEvol/lecture#section2">実習表ページに戻る</a></div>

RNA-seqを用いた解析: 遺伝子ネットワーク・転写因子・最新技術
---

　今回は、RNA-seqを用いた解析の中で、**遺伝子のネットワーク解析**というものを勉強・体験していきます。

遺伝子の発現を直接制御している**転写因子**と、転写因子が結合するDNA配列（**転写因子結合モチーフ**）の解析も体験します。

また、最近登場した新しいRNA-seqの技術として、1細胞ごとに発現を測る **single cell RNA-seq** のデータ解析も体験します。

# 今回の勉強内容
　このテキストでは、RNA-seqを用いたネットワーク解析、転写因子と結合モチーフの解析、および最近のRNA-seq技術について勉強します。

　内容は以下の通りです。

1. 遺伝子ネットワーク解析: 共発現解析
  1. 共発現している遺伝子を探す
  1. 共発現の落とし穴
  1. 発現パターンが似た遺伝子をまとめる（モジュール）
  1. ネットワーク図とハブ遺伝子
1. 転写因子と転写因子結合モチーフ
  1. 転写因子とは
  1. モチーフをデータベースから調べる（JASPAR）
  1. 配列の中からモチーフを探す
  1. モチーフの濃縮解析
  1. 転写因子の結合を調べる実験技術
1. 遺伝子ネットワーク解析: eQTL解析
  1. eQTL解析とは
  1. 自分でeQTL解析をしてみる
  1. 多数の遺伝子のeQTL解析（cis と trans）
1. 最近のRNA-seq技術
  1. single cell RNA-seq（解析の体験）
  1. Spatial Transcriptomics
  1. その他の新しい技術

　今回は、Slido（スライド）を使ったアンケートが何回かあります。🗳 Slido の印があるところでは、Slidoで回答してから次に進んでください。

---

　一度次の2つのコードセルを実行して、実習に必要なライブラリのインストールや、プログラム、サンプルファイルのダウンロードをしてください（1つ目のセルは、インストールに2〜3分かかります）。

In [ ]:
%%bash
#########################
##  実習の前に実行してください。 ##
#########################
## ソフトウェアのインストール
pip install -q pyvis                         # ネットワーク図
pip install -q pyjaspar logomaker biopython  # 転写因子結合モチーフ
pip install -q scanpy igraph leidenalg       # single cell RNA-seq
## サンプルファイルのダウンロード
wget -q https://github.com/CropEvol/lecture/raw/master/textbook_2022/datasets/RNAseq_demo.tar.gz -O - | tar -zxf - 2>/dev/null
ls RNAseq_demo

In [ ]:
#########################
##  実習の前に実行してください。 ##
#########################
import re
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set()
from scipy import stats
from IPython.display import HTML, display
from pyvis.network import Network

# 1. 遺伝子ネットワーク解析

生物はたくさんの遺伝子を持っています。(例えばイネは約30,000の遺伝子をコードしています。)

遺伝子の中には単独で働くものや、他の遺伝子と相互作用することで機能を持つものなど、様々な遺伝子があります。

遺伝子や遺伝子のコードするタンパク質の関係性を表したものをパスウェイと呼びます。

モデル生物や代表的な作物などでは、既知のパスウェイのデータベースが公開されています。

代表的なパスウェイデータベース
* [BioCyc](https://biocyc.org/)
* [KEGG](https://www.genome.jp/kegg/kegg_ja.html)
* [Reactome](https://reactome.org/)
* [WikiPathways](https://www.wikipathways.org/index.php/WikiPathways/)

### 遺伝子ネットワーク

また、遺伝子の関係性の中でも、遺伝子発現の因果関係(転写制御や促進など)をネットワークとしてモデル化するのが遺伝子のネットワーク解析になります。

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2024/images/regulate.png?raw=true" alt="network" height="120px" align="middle">

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2022/images/network.png?raw=true" alt="network" height="280px" align="middle">

遺伝子ネットワークはRNA-seqによる発現量データや、転写因子・転写因子結合部位の情報等から推定・解析されることが多いです。

共発現解析やeQTL解析と呼ばれる手法はRNAseqを用いた遺伝子ネットワークの推定手法になります。

今回はこの共発現解析とeQTL解析に加えて、転写因子と転写因子結合部位（モチーフ）の情報を使った解析を学んでいきます。

## 共発現解析

同じパスウェイ上にある機能の類似した遺伝子群や、共通の転写因子によって転写を制御されている遺伝子群などは、遺伝子の発現パターン(発現プロファイルと呼んだりします)が似ていることが多いです。

この現象を共発現と呼びます。

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2023/images/coexpression.png?raw=true" alt="coexpression" height="280px" align="middle">

様々な条件下において遺伝子の発現量を測定し、発現プロファイルを確認することで、共発現している遺伝子群を探し、同じパスウェイや転写関係に属する遺伝子群や、関連する機能を持っている遺伝子群を探すことが出来ます。

実際に共発現解析を行う場合、複数の部位やタイムポイント(ある処理をしてから1h後, 2h後, 3h後...etc)でRNA-seqを行い、遺伝子間の発現パターンを比較します。

その上で、似たような発現パターンをしている遺伝子群を共発現しているものとすることが多いです。

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2022/images/coexpression2.png?raw=true" alt="coexpression2" height="400px" align="middle">

今回は、前回扱ったイネに病原菌や水を接種した後、複数のタイムポイントでRNA-seqを行った結果を用いて共発現解析をしてみましょう。

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2022/images/coexpression3.png?raw=true" alt="coexpression3" height="400px" align="middle">

複数のタイムポイントでRNA-seqを行い、発現量をTMM値に変換したデータを下のコードで読み込みます。

In [ ]:
# 共発現解析のためのデータ
TMM = pd.read_csv("RNAseq_demo/TMM_coex.csv", index_col=0)
TMM

この様なデータを元に、ある遺伝子と別の遺伝子が共発現しているかを確認します。

何をもって共発現とするのかという所ですが、

最もシンプルな確認方法としては、着目した２つの遺伝子の発現パターンの間で相関係数をとり、1に近いほど共発現しているものとする、という指標になります。

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2022/images/coexpression4.png?raw=true" alt="coexpression4" height="250px" align="middle">

ということで、今回のデータに基づいて、

`Gene.2`という遺伝子と共発現していそうな遺伝子を探してみましょう。

下記コードは`Gene.2`と`Gene_B = Gene.XXX`で指定した遺伝子の発現パターンの相関係数を示してくれます。

In [ ]:
# 遺伝子Gene.2と共発現している遺伝子を探す
Gene_A = "Gene.2"
Gene_B = "Gene.3"

Gene_A_exp = TMM.loc[Gene_A, :]
Gene_B_exp = TMM.loc[Gene_B, :]

print(np.corrcoef([Gene_A_exp, Gene_B_exp])[0, 1])

相関係数が高そうな遺伝子の組み合わせを見つけたら、

実際に発現パターンをグラフ化して確認してみます。

下記コードには相関係数が`0.99`の組み合わせを図示していますが、自分で他に良さそうな組み合わせを見つけた場合は、`Gene_B = Gene.XXXX`の所を書き換えて確認してみてください。

In [ ]:
# 発現量をプロットして確認してみる
Gene_A = "Gene.2"
Gene_B = "Gene.19974"
plt.plot(TMM.loc[Gene_A, :], marker='o', label=Gene_A)
plt.plot(TMM.loc[Gene_B, :], marker='o', label=Gene_B)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0, fontsize=11)
plt.xticks(rotation=60)
plt.show()

### 練習1-1: 共発現している遺伝子をまとめて探す

　上のように1つずつ相関係数を調べるのは大変です。次のコードセルでは、`target` に指定した遺伝子と、**すべての遺伝子**との相関係数を一度に計算し、相関係数が高い順に並べます。

　ほとんど発現していない遺伝子は、偶然高い相関係数になりやすいので、平均の発現量（TMM）が5以上の遺伝子だけを使います。また、発現量は桁が大きく違うので、log<sub>2</sub>(TMM + 1) に変換してから計算します。

　`target` を `"Gene.2"` 以外の遺伝子（例: `"Gene.7"`、`"Gene.12"`）にも変えて試してみましょう。

In [ ]:
target = "Gene.2"   # 共発現している遺伝子を探したい遺伝子

## 発現している遺伝子だけを使い、log2(TMM+1) に変換する
expressed = TMM[TMM.mean(axis=1) >= 5]
log_TMM = np.log2(expressed + 1)

## target とすべての遺伝子の相関係数を計算し、高い順に並べる
corr = log_TMM.T.corrwith(log_TMM.loc[target])
ranking = corr.drop(target).sort_values(ascending=False)
print(f"{target} と相関係数が高い遺伝子（上位10）")
print(ranking.head(10))

## 上位3つの遺伝子の発現パターンを描く
plt.figure(figsize=(8, 4))
for gene in [target] + list(ranking.index[:3]):
    plt.plot(log_TMM.loc[gene], marker="o", label=gene, lw=3 if gene == target else 1)
plt.ylabel("log2(TMM + 1)")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.xticks(rotation=60)
plt.show()

## 共発現の落とし穴

### 落とし穴1: 相関係数が高い遺伝子が多すぎる？

このやり方だと、Gene.2 と相関係数0.9以上の遺伝子は2,000個以上もあります。

本当に全ての遺伝子と関係しているでしょうか？

下のコードでは、発現量をランダムに並べ替えて意味がないデータにした場合、共発現に見えてしまう遺伝子の数を確認してみます。

In [ ]:
target = "Gene.2"
n_high = (corr.drop(target) >= 0.9).sum()
print(f"{target} と相関係数 0.9 以上の遺伝子: {n_high} 個")

## 比較: 各遺伝子の8つの値をランダムに並べ替え、発現パターンを壊したデータで同じことをする
rng = np.random.default_rng(0)
shuffled = pd.DataFrame([rng.permutation(v) for v in log_TMM.values], index=log_TMM.index, columns=log_TMM.columns)
corr_shuffled = shuffled.T.corrwith(log_TMM.loc[target])
print(f"並べ替えたデータで相関係数 0.9 以上の遺伝子: {(corr_shuffled.drop(target) >= 0.9).sum()} 個")

　並べ替えたデータ（偶然だけの場合）では数十個なので、少なくとも偶然では無いようです。

今回のデータでは、時間の経過（16h → 72h）にともない8観測点で同じように増えたり減ったりする遺伝子がとても多く、そのような遺伝子どうしがすべて「共発現」に見えてしまっている形になります。

　サンプル（条件）の数が少ないと、「同じ制御を受けている遺伝子」と「たまたま時間とともに同じ方向に変化しただけの遺伝子」を区別できません。そのため、実際の共発現解析では、数十〜数百の条件のRNA-seqデータを使います。たくさんの公開データをまとめた共発現データベース（例: [ATTED-II](https://atted.jp/)）もあります。

### 落とし穴2: 外れ値に引っ張られる

　単純な相関係数を基準にすることに問題がある場合も多いです。

例えば下図２段目の様な時です。

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2022/images/coexpression5.png?raw=true" alt="coexpression5" height="500px" align="middle">

1段目の例は共発現していると言ってよさそうですが、2段目の例では、相関係数としては高い値になっていますが、60h後のデータに引っ張られているだけで、0~48h後のデータは全く相関していないことが分かるかと思います。

単純に相関係数を指標とすると、この様な遺伝子のペアも共発現しているものとしてカウントされるので、注意が必要です。

　1つの時点の値に引っ張られて相関係数が高くなるような場合には、値そのものではなく**値の順位**を使って計算する **スピアマンの順位相関係数** や、Mutual Rank(Obayashi et al., 2009)など様々な指標が考案されていますが、共発現を決める絶対的な指標は存在しておらず、何かの指標で検出されたペアがどんな発現プロファイルか、本当に共発現の関係にあるか、は都度確認しておけると良いでしょう。

## 発現パターンが似た遺伝子をまとめる（モジュール）

　1つの遺伝子に注目するのではなく、ゲノム全体の遺伝子を、発現パターンの似たグループに分けることもできます。このグループを**共発現モジュール**と呼びます。同じモジュールの遺伝子は、同じ転写因子に制御されていたり、同じパスウェイで働いていたりする可能性があります。

　ここでは、発現量の変化が大きい上位2,000遺伝子を、階層的クラスタリングで `n_modules` 個のモジュールに分けます。遺伝子ごとに発現量を標準化（平均0、標準偏差1）しているので、発現量の大きさではなく「パターン」で分けられます。

### 練習1-3
　`n_modules` を 4 → 6 → 8 と変えて、どのようなパターンのモジュールが現れるか見てみましょう。水（青）と病原菌（赤）で、パターンが違うモジュールはありますか？

In [ ]:
from scipy.cluster.hierarchy import linkage, fcluster

n_modules = 4   # モジュールの数

## 発現量の変化（分散）が大きい上位2000遺伝子を、遺伝子ごとに標準化する
top_genes = log_TMM.var(axis=1).sort_values(ascending=False).index[:2000]
z = log_TMM.loc[top_genes]
z = z.sub(z.mean(axis=1), axis=0).div(z.std(axis=1), axis=0)

## 階層的クラスタリングでモジュールに分ける
modules = pd.Series(fcluster(linkage(z.values, method="ward"), n_modules, criterion="maxclust"), index=z.index)

## モジュールごとの発現パターン（細線: 各遺伝子、太線: 平均）
times = ["16h", "24h", "48h", "72h"]
fig, axes = plt.subplots(1, n_modules, figsize=(3 * n_modules, 3), sharey=True)
for ax, m in zip(axes, sorted(modules.unique())):
    sub = z[modules == m]
    ax.plot(times, sub.iloc[:, :4].T.values, color="tab:blue", alpha=0.03)
    ax.plot(times, sub.iloc[:, 4:].T.values, color="tab:red", alpha=0.03)
    ax.plot(times, sub.iloc[:, :4].mean().values, color="tab:blue", marker="o", lw=3, label="water")
    ax.plot(times, sub.iloc[:, 4:].mean().values, color="tab:red", marker="o", lw=3, label="pathogen")
    ax.set_title(f"Module {m} ({len(sub)} genes)")
axes[0].set_ylabel("z-score"); axes[0].legend()
plt.tight_layout(); plt.show()

#### 確認ポイント
- 遺伝子の多いモジュールの多くは、水でも病原菌でも同じように、時間とともに増える（または減る）パターンでした。これは「落とし穴1」で見た、時間にともなう共通の変化です。
- 一方、水と病原菌でパターンが違うモジュール（病原菌で発現が下がるものなど）もあります。病原菌に対する応答に関わる遺伝子群の候補になります。

　実際の研究では、[WGCNA](https://horvath.genetics.ucla.edu/html/CoexpressionNetwork/Rpackages/WGCNA/)（R）などのツールで、より多くのサンプルを使ってモジュールを検出します。

共発現していると考えられる遺伝子群を特定した後は、その情報をネットワーク図として描写することが多いです。

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2022/images/coexpression6.png?raw=true" alt="coexpression6" height="250px" align="middle">

先ほどのモジュールの1つを使って、遺伝子ネットワークを描いてみます。

ネットワーク図を描くパッケージやライブラリにはRの`igraph`等、様々なものがあります

今回は`pyvis`というPythonのライブラリを使用してネットワーク図を描いてみます。

　次のコードセルでは、`module` で選んだモジュールの遺伝子のうち、相関係数が `threshold` 以上の遺伝子のペアを線（エッジ）で結んだネットワーク図を描きます。ネットワーク図は、マウスでドラッグして動かしたり、拡大したりできます。

　また、たくさんの遺伝子とつながっている遺伝子を**ハブ遺伝子**と呼びます。ハブ遺伝子は、モジュール全体の発現を制御する転写因子などの、重要な遺伝子である可能性があります。

In [ ]:
# network図の描写
module = 6          # 描くモジュールの番号（上の図のタイトルの番号）
threshold = 0.98    # 線で結ぶ相関係数の基準
max_genes = 150     # 描く遺伝子の数の上限（多すぎると重くなるため）

genes = modules[modules == module].index[:max_genes]
c = log_TMM.loc[genes].T.corr()          # モジュール内の遺伝子どうしの相関係数

net = Network(height="500px", width="100%", notebook=True, cdn_resources="in_line")
degree = {}
for i, g1 in enumerate(genes):
    for g2 in genes[i + 1:]:
        if c.loc[g1, g2] >= threshold:
            net.add_node(g1, label=g1); net.add_node(g2, label=g2)
            net.add_edge(g1, g2)
            degree[g1] = degree.get(g1, 0) + 1; degree[g2] = degree.get(g2, 0) + 1

print(f"モジュール{module}: 遺伝子 {len(degree)} 個、つながり {sum(degree.values()) // 2} 本")
print("つながりの多い遺伝子（ハブ遺伝子の候補）:")
print(pd.Series(degree).sort_values(ascending=False).head(5))

net.write_html("network.html")
display(HTML(open("network.html").read()))

上にネットワーク図が表示されたでしょうか。表示されない場合は、Google Colab内のファイルに生成された`network.html`というファイルをダウンロードしてブラウザで開くと、ネットワーク図が描写されます。`module` や `threshold` を変えて、ネットワークの形がどう変わるか見てみましょう。

今回は1つのモジュールの一部の遺伝子をネットワークにしただけですが、遺伝子は他にもたくさんあります。

ネットワーク解析を行う際は、一気にすべての遺伝子をネットワークにしようとすると、解析時間が異常にかかってしまったり、図が複雑になりすぎてしまいます。

そのため、着目する遺伝子や遺伝子群に絞って見始める等の工夫が必要です。

例) 菌感染時に発現量が上昇している遺伝子だけを使ってネットワーク図を描いてみる、等

# 2. 転写因子と転写因子結合モチーフ

## 転写因子とは

　共発現解析で「同じように発現する遺伝子群」が見つかったとき、次に知りたくなるのは「**何が、これらの遺伝子の発現をまとめて制御しているのか？**」です。その答えの多くは**転写因子**（Transcription Factor, TF）です。

　転写因子は、遺伝子の上流（プロモーター領域）にある特定の短いDNA配列（**シス配列**、cis-element）に結合し、その遺伝子の転写を促進したり抑制したりするタンパク質です。1つの転写因子が、同じシス配列を持つたくさんの遺伝子をまとめて制御するので、それらの遺伝子は共発現します。


<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2026/images/TFs.png?raw=true" alt="coexpression6" height="250px" align="middle">


　植物は、全遺伝子の数%にあたる、数千個の転写因子をもっています。転写因子は、DNAに結合する部分（DNA結合ドメイン）の構造によってファミリーに分けられ、ファミリーごとに結合する配列の特徴が似ています。

|ファミリー|結合する配列の例|植物での主な役割の例|
|:--|:--|:--|
|WRKY|W-box（TTGACC/T）|病原菌に対する防御応答|
|ERF/DREB（AP2/ERF）|GCC-box（GCCGCC）、DRE（CCGAC）|エチレン応答、乾燥・低温ストレス応答|
|NAC|CGT[GA] を含む配列|ストレス応答、老化、二次細胞壁の形成|
|MYB|AC-element など|二次代謝（色素など）、細胞の分化|
|bZIP|G-box（CACGTG）、ABRE|アブシジン酸（ABA）応答、光応答|

　たとえばイネでは、WRKYファミリーの転写因子 **OsWRKY45** が、いもち病に対する抵抗性に重要な役割を果たすことが知られています。前回のRNA-seqで病原菌によって発現が上がった遺伝子の上流にも、WRKYが結合するW-boxが多いかもしれません。

　転写因子が結合する配列は、1種類に決まっているわけではなく、少しずつ違う配列にも結合します。そのため、結合する配列のパターンをまとめて、**転写因子結合モチーフ**（以下、モチーフ）として表します。

## モチーフをデータベースから調べる（JASPAR）

　転写因子が結合する配列は、実験によって調べられ、データベースに登録されています。代表的なデータベースが [JASPAR](https://jaspar.elixir.no/) です。ここでは、JASPARのデータをPythonから使える `pyjaspar` というライブラリで、植物の転写因子のモチーフを見てみましょう。

In [ ]:
from pyjaspar import jaspardb

jdb = jaspardb()                                                    # JASPARのデータベース
plant_motifs = jdb.fetch_motifs(collection="CORE", tax_group=["plants"])
print(f"{jdb.release}: 植物の転写因子のモチーフ {len(plant_motifs)} 個")

## 転写因子ファミリーごとのモチーフの数
families = pd.Series([f for m in plant_motifs for f in (m.tf_family or ["unknown"])])
families.value_counts().head(10).plot.barh()
plt.gca().invert_yaxis()
plt.xlabel("Number of motifs")
plt.show()

### モチーフの表し方

　モチーフは、「各位置に、A・C・G・T がそれぞれ何回現れたか」を数えた表（**塩基頻度行列**）で表されます。シロイヌナズナの WRKY12 のモチーフ（MA1075.1）を見てみましょう。

In [ ]:
motif = jdb.fetch_motif_by_id("MA1075.1")     # WRKY12（シロイヌナズナ）
print("転写因子:", motif.name, " ファミリー:", motif.tf_family, " 生物種(Taxonomy ID):", motif.species)
print("コンセンサス配列（各位置で最も多い塩基）:", motif.counts.consensus)
counts_table = pd.DataFrame({base: list(motif.counts[base]) for base in "ACGT"})
counts_table

　表の行が位置（0〜7）、列が塩基です。たとえば位置2では T が913回、位置5では A が971回で、ほぼ決まった塩基が現れています。一方、位置0や位置7では、いくつかの塩基が混ざっています。

　この表を図にしたものが**配列ロゴ**です。各位置の文字の高さは、その位置の塩基がどれくらい決まっているか（**情報量**、最大2ビット）を表します。

In [ ]:
import logomaker

def draw_logo(motif):
    """JASPARのモチーフから配列ロゴを描く"""
    counts_table = pd.DataFrame({base: list(motif.counts[base]) for base in "ACGT"})
    info = logomaker.transform_matrix(counts_table, from_type="counts", to_type="information")
    logo = logomaker.Logo(info, figsize=(0.6 * len(info) + 1, 2))
    logo.ax.set_ylabel("bits"); logo.ax.set_ylim(0, 2)
    logo.ax.set_title(f"{motif.matrix_id} {motif.name} ({', '.join(motif.tf_family or [])})")
    plt.show()

draw_logo(motif)

　中央の **TTGAC** の部分が大きく描かれていて、WRKYが結合するW-box（TTGACC/T）の特徴がよくわかります。

### 練習2-1
　他の転写因子のモチーフも見てみましょう。次のコードセルの `name` を変えて実行してください。JASPARには、イネの転写因子のモチーフも登録されています。

- `"OsCCA1"`（イネ、概日時計に関わるMYB-related）
- `"OsERF104"`（イネ、ERF）
- `"OsbZIP42"`（イネ、bZIP）
- `"WRKY51"`, `"NAC055"`, `"MYB46"`（シロイヌナズナ）

In [ ]:
name = "OsCCA1"   # 調べたい転写因子の名前

for m in jdb.fetch_motifs_by_name(name):
    print(m.matrix_id, m.name, "コンセンサス配列:", m.counts.consensus)
    draw_logo(m)

#### 確認ポイント
- 同じファミリー（例: WRKY12 と WRKY51）の転写因子は、似たモチーフをもっていましたか？
- モチーフの長さや、文字の高さ（どれくらい配列が決まっているか）は、転写因子によってどう違いましたか？

## 配列の中からモチーフを探す

　モチーフが分かれば、遺伝子のプロモーター配列の中に、その転写因子が結合しそうな場所があるかを探せます。

### 方法1: コンセンサス配列で探す

　最も簡単なのは、コンセンサス配列（W-boxなら TTGACC または TTGACT）と一致する場所を探す方法です。DNAは2本鎖なので、**逆相補鎖**（反対側の鎖）も調べる必要があります。TTGAC[CT] の逆相補鎖は [AG]GTCAA です。

　Pythonでは、`re`（正規表現）を使うと、`[CT]` のように「CまたはT」を含むパターンで検索できます。

In [ ]:
sequence = "AATTTGACCAAGGTCAACGTTGACCTTAGCTTGACTAAAT"   # 調べたい配列

## W-box（TTGAC[CT]）と、その逆相補鎖（[AG]GTCAA）を探す
for match in re.finditer(r"(?=(TTGAC[CT]))", sequence):
    print(f"位置 {match.start():>3}（+鎖）: {match.group(1)}")
for match in re.finditer(r"(?=([AG]GTCAA))", sequence):
    print(f"位置 {match.start():>3}（−鎖）: {match.group(1)}")

### 方法2: 塩基頻度行列（PWM）で点数をつける

　コンセンサス配列での検索は、1塩基でも違うと見つけられません。しかし、先ほどモチーフで見たように、転写因子は少しずつ違う配列にも結合します。塩基頻度行列から「各位置にその塩基が来るとどれくらいモチーフらしいか」の点数表（**位置重み行列**、Position Weight Matrix、**PWM**）を作り、配列の各位置に点数をつける方法がよく使われます。点数が高い場所ほど、転写因子が結合しやすいと予測されます。

In [ ]:
from Bio.Seq import Seq

## WRKY12 のモチーフから PWM（点数表）を作る
## 背景の塩基組成は、イネのプロモーターに多い AT が多めの組成にする
pwm = motif.counts.normalize(pseudocounts=0.5).log_odds({"A": 0.31, "T": 0.31, "G": 0.19, "C": 0.19})
print(f"最高点: {pwm.max:.1f}, 最低点: {pwm.min:.1f}")

## 配列の中で、最高点の70%以上の点数になる場所を探す
seq = Seq(sequence)
for pos, score in pwm.search(seq, threshold=pwm.max * 0.7):
    strand = "+" if pos >= 0 else "−"
    start = pos if pos >= 0 else len(seq) + pos      # −鎖の場合は位置を変換する
    print(f"位置 {start:>3}（{strand}鎖）: {seq[start:start + len(pwm.consensus)]}  点数 {score:.1f}")

## モチーフの濃縮解析

　では遺伝子のプロモーター領域にWRKY転写因子の結合するW-box（TTGACC/T）配列があれば、その遺伝子はWRKYに制御されている、と言えるでしょうか？

下のコードは適当な配列を作成し、W-box（TTGACC/T）配列が見つかる確率を計算します。

In [ ]:
rng = np.random.default_rng(1)

def random_sequence(length):
    """AT が多め（イネのプロモーターに近い組成）の、でたらめな配列を作る"""
    return "".join(rng.choice(list("ATGC"), length, p=[0.31, 0.31, 0.19, 0.19]))

def count_wbox(seq):
    """W-box（両方の鎖）の数を数える"""
    return len(re.findall(r"(?=(TTGAC[CT]|[AG]GTCAA))", seq))

counts_random = [count_wbox(random_sequence(1000)) for _ in range(1000)]
print(f"でたらめな1,000塩基に含まれるW-boxの数: 平均 {np.mean(counts_random):.2f} 個")
print(f"W-boxを1つ以上含む配列の割合: {np.mean(np.array(counts_random) >= 1) * 100:.0f}%")

　でたらめな配列でも、1,000塩基あたり平均1個くらいW-boxが現れ、約7割の配列が1つ以上のW-boxを含んでいました。

当然ですが、6塩基ほどの短い配列は、偶然にもよく現れます。そのため、「プロモーターにW-boxがある」だけでは、WRKYに制御されている証拠にはなりません。

　そこで、**共発現モジュールなど注目する遺伝子群のプロモーターに、他の遺伝子より多くモチーフが含まれているか**を統計的に調べます。これを**モチーフの濃縮解析**（モチーフエンリッチメント解析）と呼びます。

### 濃縮解析をやってみる

- 注目する遺伝子群: 「病原菌で発現が上がった」100遺伝子
- 比較対象: それ以外の400遺伝子

これらの遺伝子セットに対してモチーフのエンリッチメント解析を実施すると、WRKYの結合するモチーフはどちらが多いかを確認することができます。(今回は仮データで実施します。)

In [ ]:
##プロモーター配列を作る（本来はゲノム配列から取り出す）
rng = np.random.default_rng(2026)
def plant_motif(seq, n):
    """配列のランダムな位置に、W-box を n 個書き込む"""
    seq = list(seq)
    for _ in range(n):
        m = rng.choice(["TTGACC", "TTGACT"])
        if rng.random() < 0.5:
            m = m[::-1].translate(str.maketrans("ATGC", "TACG"))   # 逆相補鎖にする
        p = rng.integers(0, len(seq) - len(m))
        seq[p:p + len(m)] = m
    return "".join(seq)

module_promoters = [plant_motif(random_sequence(1000), rng.poisson(1.5)) for _ in range(100)]   # 注目する遺伝子群
other_promoters = [random_sequence(1000) for _ in range(400)]                                  # それ以外の遺伝子

def count_pattern(seq, pattern):
    return len(re.findall(f"(?=({pattern}))", seq))

pattern = "TTGAC[CT]|[AG]GTCAA"   # 調べるモチーフ（W-box、両方の鎖）
min_count = 2                     # 「モチーフを持つ」とする最低の個数

module_n = [count_pattern(s, pattern) for s in module_promoters]
other_n = [count_pattern(s, pattern) for s in other_promoters]
a = sum(n >= min_count for n in module_n); b = len(module_n) - a
c = sum(n >= min_count for n in other_n); d = len(other_n) - c
odds, p = stats.fisher_exact([[a, b], [c, d]])

print(f"モチーフを{min_count}個以上もつ遺伝子の割合")
print(f"  注目する遺伝子群: {a}/{len(module_n)}（{a / len(module_n) * 100:.0f}%）")
print(f"  それ以外        : {c}/{len(other_n)}（{c / len(other_n) * 100:.0f}%）")
print(f"フィッシャーの正確確率検定: p = {p:.1e}")

plt.hist([module_n, other_n], bins=range(0, 9), density=True, label=["module genes", "other genes"])
plt.xlabel("Number of motifs in promoter"); plt.ylabel("Fraction of genes"); plt.legend()
plt.show()

　注目する遺伝子群では、W-boxを2個以上もつ遺伝子の割合が、それ以外の遺伝子より明らかに高く、p値も非常に小さくなりました。このような結果が実際のデータで得られれば、「このモジュールの遺伝子は、WRKYファミリーの転写因子にまとめて制御されているかもしれない」と考えられます。そこで、モジュールの中のWRKY遺伝子や、共発現ネットワークのハブになっているWRKY遺伝子が、制御因子の有力な候補になります。

 実際の研究では、既知のモチーフをまとめて調べるツール（[MEME Suite](https://meme-suite.org/) の SEA や AME、[HOMER](http://homer.ucsd.edu/homer/) など）や、未知のモチーフを配列から探し出す（de novo モチーフ探索）ツール（MEME Suite の MEME や STREME など）がよく使われます。植物の転写因子とモチーフの情報は、[PlantTFDB](https://planttfdb.gao-lab.org/) や [PlantPAN](http://plantpan.itps.ncku.edu.tw/) でも調べられます。

## 転写因子の結合を調べる実験技術

　モチーフの濃縮解析は、あくまで「結合しそうな配列がある」という予測です。転写因子が実際にどこに結合しているかは、シーケンス技術を使った実験で調べられます。

- **ChIP-seq**（クロマチン免疫沈降シーケンス）: 細胞の中で転写因子が結合しているDNAを、抗体で回収してシーケンスする。生体内での結合がわかるが、転写因子ごとに抗体が必要
- **DAP-seq**（DNA affinity purification sequencing）: 試験管内で作った転写因子に、ゲノムDNAの断片を結合させて回収し、シーケンスする。抗体が不要で、多数の転写因子を一度に調べられる。シロイヌナズナでは500以上の転写因子の結合部位がまとめて報告された（O'Malley et al. 2016, *Cell* 165: 1280–1292）。JASPARに登録されている植物のモチーフの多くも、DAP-seqやPBM（タンパク質結合マイクロアレイ）のデータから作られている
- **ATAC-seq**: DNAがほどけていて転写因子が結合しやすい領域（オープンクロマチン）をシーケンスで調べる

　共発現（どの遺伝子が一緒に動くか）、モチーフ（どの転写因子が結合しそうか）、ChIP-seq/DAP-seq（実際にどこに結合するか）、そして次のeQTL（どの遺伝子座が発現を変えるか）の情報を組み合わせることで、遺伝子の制御関係（遺伝子制御ネットワーク）を推定していきます。

# 3. 遺伝子ネットワーク解析: eQTL解析

共発現解析では、似たような発現プロファイルの遺伝子群を発現量のデータから探すことで、同じパスウェイや制御関係にある遺伝子群を探しました。

しかし、共発現解析では、遺伝子群の中でどの遺伝子が上流なのか下流なのかということを決めることは出来ません。

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2022/images/coexpression7.png?raw=true" alt="coexpression7" height="200px" align="middle">

勿論、既に転写因子としての機能を持つことが知られている様な遺伝子であれば、制御関係を決めることは出来そうですが、大半の遺伝子はまだ機能が分かっていないので現実的ではありません。

そういった場合に、１つの情報を提供する手法となるのがeQTL解析になります。

eQTL解析は少し前にやったQTLの探索をRNA-seqに応用したものです。

QTLの探索がどんなものだったかというと、交配集団の形質・遺伝子型データ等を用いて、ある形質に影響を与える遺伝子座(量的形質遺伝子座:QTL)を探索する解析でした。

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2022/images/QTL_example.png?raw=true" alt="QTL_example" height="280px" align="middle">

eQTL解析も同じようにQTLを探す解析ですが、これまでのQTL解析と大きく異なる点は、

対象とする形質が葉の長さや籾数等の形質ではなく、遺伝子の発現量を対象にする。という所です。

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2022/images/eQTL1.png?raw=true" alt="eQTL1" height="400px" align="middle">

つまり、これまでのQTL解析では葉の長さや籾数などに影響を与えている遺伝子座を検出することが目的でしたが、

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2024/images/QTL_normal.png?raw=true" alt="eQTL2" height="200px" align="middle">

eQTL解析では、ある特定の遺伝子の発現量に影響を与えている遺伝子座を検出することが目的になります。

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2024/images/eQTL_normal.png?raw=true" alt="eQTL2" height="600px" align="middle">

### 必要なデータ

eQTL解析を行うには、以前やったGWAS等と同じ様に、集団の遺伝子型のデータと、形質値のデータが必要です。

普通のGWASだと、系統毎に各形質のデータがありましたが、

eQTL解析だと、この形質値のデータが葉の長さ等から遺伝子の発現量に変わるという訳です。

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2023/images/eQTL_phenotype.png?raw=true" alt="eQTL_phenotype" height="400px" align="middle">

今回はイネのひとめぼれという系統とKALUHEENATIという系統を交配して作成した集団を用いて、eQTL解析をしてみましょう。

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2022/images/eQTL3.png?raw=true" alt="eQTL3" height="300px" align="middle">

In [ ]:
#eQTL データ
TMM_eQTL = pd.read_csv("RNAseq_demo/TMM_eQTL.csv", index_col=0)
TMM_eQTL

以前GWASを扱ったかと思います。その時は例えば籾数であったり葉の長さを形質として解析を行いましたが、

eQTL解析では、遺伝子の発現量を形質としてQTL解析を行う形になります。

例えば7番染色体に位置する`Gene.22078`という遺伝子の発現量を形質としてGWASを行ってみましょう。

`Gene.22078`という遺伝子の発現量に影響を与えている遺伝子座(eQTL)を検出することが出来ます。


In [ ]:
# GWAS結果
eQTL_22078 = pd.read_csv("RNAseq_demo/eQTL_22078.csv", index_col=0)
plt.figure(figsize=(12, 3))
sns.scatterplot(x="Marker", y="Score", hue="chr", data=eQTL_22078, s=10, linewidth=0, legend=False, palette=["blue", "green"]*6)
plt.ylabel("-log10(p)")
plt.show()

In [ ]:
print("特に強く検出された領域")
display(eQTL_22078.loc[eQTL_22078.chr == "chr07", ].sort_values(by="Score",ascending=False).head(3))
display(eQTL_22078.loc[eQTL_22078.chr == "chr10", ].sort_values(by="Score",ascending=False).head(3))

この結果から、`Gene.22078`という遺伝子の発現量は、7番染色体の20Mb付近と、10番染色体の7Mb付近のeQTLの遺伝子型によって大きく変化することが分かります。一応Boxplotを描いてみてみると

In [ ]:
# boxplot
genotype = pd.read_csv("RNAseq_demo/eQTL_genotype.csv", index_col=0)
boxplot_data = pd.DataFrame({
    "TMM":TMM_eQTL.loc["Gene.22078", :].values,
    "Genotype1":genotype.loc[:, TMM_eQTL.columns].iloc[0, :],
    "Genotype2":genotype.loc[:, TMM_eQTL.columns].iloc[1, :],
})
boxplot_data = boxplot_data[(boxplot_data.Genotype1 != 1) & (boxplot_data.Genotype2 != 1)]

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
for col, ax in zip([1, 2], axes):
    p = sns.boxplot(x="Genotype{}".format(col), y="TMM", data=boxplot_data, ax=ax)
    p.set_title("{}_{}".format(genotype.iloc[col-1, 1], genotype.iloc[col-1, 2]))
    p = sns.stripplot(x="Genotype{}".format(col), y='TMM', data=boxplot_data, jitter=True, color='black', ax=ax)
    p.set_xticklabels(['Hitomebore', 'Kaluheenati'])

この様に、検出されたeQTLの遺伝子型が変わると、発現量が変化することが分かります。

`Gene.22078`という遺伝子は7番染色体の20255488bpに位置しているので、形質とした遺伝子の非常に近い位置と、別の染色体にピークが検出されたことになります。

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2022/images/eQTL_pos.png?raw=true" alt="eQTL_pos" height="280px" align="middle">

eQTLは通常のQTLとは異なり、cis-eQTLとtrans-eQTLという２種類のeQTLに分類されます。

cis-eQTLは対象としている遺伝子近傍に位置しているeQTLです。

これは、他の遺伝子による影響ではなく、自分自身のプロモーター領域等に変異が入り、発現量が大きく変わっていることを表しています。

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2022/images/cis_eQTL.png?raw=true" alt="cis_eQTL" height="280px" align="middle">

trans-eQTLは対象としている遺伝子と離れた位置にあるeQTLです。

これは、他の遺伝子による影響で発現量が大きく制御されており、その遺伝子に変異が入ることで制御機構が変化したことで起きていると考えられます。

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2022/images/trans_eQTL.png?raw=true" alt="trans_eQTL" height="280px" align="middle">

つまり、今回得られた結果は`Gene.22078`という遺伝子の発現量を制御する遺伝子が10番染色体の7Mb付近に位置することを示唆しています。この領域に転写因子をコードする遺伝子等が位置していれば、可能性は高そうですね。

eQTL解析によって得られる結果は制御関係の上流下流がはっきり決まっているので、共発現解析の時とは異なり、どの遺伝子がどの遺伝子を制御しているのか、という情報に近付きやすいのが利点になります。

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2022/images/eQTL_benefit.png?raw=true" alt="eQTL_benefit" height="180px" align="middle">

ただし、検出できるのはあくまでも遺伝子座であって遺伝子単体では無いので、検出された領域から候補となる制御遺伝子を見つける方法を別途考える必要はあります。

更に、今回は`Gene.22078`という遺伝子のみの解析結果を見ましたが、イネは約30,000の遺伝子があります。そのため、30,000遺伝子それぞれの発現量に対しeQTL解析を行い、その結果をまとめて考える必要がある、かなり規模の大きな解析になります。

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2023/images/eQTL_phenotype2.png?raw=true" alt="eQTL_phenotype2" height="400px" align="middle">

## 自分でeQTL解析をしてみる

　上の `Gene.22078` の結果は、あらかじめ計算しておいたものでした。ここでは、集団の遺伝子型データ（約48,000マーカー）と発現量データを使って、自分でeQTL解析をしてみましょう。GWASの回の「単純な回帰分析（t検定と同じ）」と同じ方法で、全マーカーについて、遺伝子型と発現量（log<sub>2</sub>(TMM + 1)）の関係を検定します。

　まず、遺伝子型データと、各遺伝子のゲノム上の位置の情報を読み込み、eQTL解析をおこなう関数を準備します（中身を理解する必要はありません）。

In [ ]:
## 遺伝子型データ（行: マーカー、列: 系統）。発現量データの系統名（Sample001など）に合わせる
geno = pd.read_csv("RNAseq_demo/N09_imputed_minDP5_genotype_remove_outlier_5000bp_skipped.csv")
samples = [s.replace("Sample", "N09_") for s in TMM_eQTL.columns]
X = geno[samples].values.T.astype(float)                      # 系統 × マーカー
X = np.where(np.isnan(X), np.nanmean(X, axis=0), X)           # 欠測値はマーカーの平均で埋める
use = X.std(axis=0) > 0                                       # 遺伝子型が1種類しかないマーカーは除く
X, markers = X[:, use], geno.loc[use, ["Marker", "Chrom", "Position"]].reset_index(drop=True)
Xc = X - X.mean(axis=0); sxx = (Xc ** 2).sum(axis=0)

## 各遺伝子のゲノム上の位置（前回の featureCounts の結果から）
fc = pd.read_csv("RNAseq_demo/feature_counts.txt", sep="\t", comment="#", usecols=["Geneid", "Chr", "Start"])
gene_pos = pd.DataFrame({"chr": fc["Chr"].str.split(";").str[0],
                         "pos": fc["Start"].astype(str).str.split(";").str[0].astype(int)}).set_index(fc["Geneid"])

def eqtl_scan(Y):
    """発現量（系統 × 遺伝子）と全マーカーの関係を回帰分析で検定し、-log10(p) を返す（マーカー × 遺伝子）"""
    Yc = Y - Y.mean(axis=0); n = Y.shape[0]
    B = Xc.T @ Yc
    beta = B / sxx[:, None]
    rss = (Yc ** 2).sum(axis=0)[None, :] - beta * B
    t = beta / np.sqrt(rss / (n - 2) / sxx[:, None])
    return -(np.log(2) + stats.t.logsf(np.abs(t), n - 2)) / np.log(10)

def classify(gene, eqtl_chr, eqtl_pos, window=2_000_000):
    """eQTLが遺伝子の近く（同じ染色体で2 Mb以内）なら cis、それ以外は trans"""
    g = gene_pos.loc[gene]
    return "cis" if (g["chr"] == eqtl_chr and abs(g["pos"] - eqtl_pos) <= window) else "trans"

print(f"系統数: {X.shape[0]}, マーカー数: {X.shape[1]}")

### 練習3-1
　次のコードセルで、`gene` に指定した遺伝子のeQTL解析をおこないます。まずは `"Gene.22078"` で実行して、上の結果と同じように7番染色体と10番染色体にピークが出るか確認してください。その後、他の遺伝子（例: `"Gene.2"`、`"Gene.12504"`、`"Gene.7"`）でも試してみましょう。

In [ ]:
gene = "Gene.22078"   # eQTL解析をおこなう遺伝子

y = np.log2(TMM_eQTL.loc[[gene]].values.T + 1)       # 発現量（系統 × 1遺伝子）
score = eqtl_scan(y)[:, 0]
threshold = -np.log10(0.05 / len(score))               # Bonferroni補正の基準

## マンハッタンプロット（★: 遺伝子自身の位置）
result = markers.assign(score=score)
colors = np.where(result["Chrom"].str[-2:].astype(int) % 2 == 0, "tab:green", "tab:blue")
plt.figure(figsize=(12, 3))
plt.scatter(result.index, result["score"], s=3, c=colors)
plt.axhline(threshold, color="red", ls="--", lw=1)
g = gene_pos.loc[gene]
own = result[(result["Chrom"] == g["chr"])]
own_index = (own["Position"] - g["pos"]).abs().idxmin()
plt.scatter(own_index, 0, marker="*", s=300, color="orange", zorder=5)
plt.title(f"{gene} ({g['chr']}: {g['pos']:,} bp)"); plt.ylabel("-log10(p)"); plt.xticks([])
plt.show()

## 各染色体で最も強いピークと、cis / trans の判定
peaks = result.loc[result.groupby("Chrom")["score"].idxmax()]
peaks = peaks[peaks["score"] >= threshold].sort_values("score", ascending=False)
peaks["type"] = [classify(gene, c, p) for c, p in zip(peaks["Chrom"], peaks["Position"])]
print(f"基準値（-log10(p)）: {threshold:.1f}")
print(peaks if len(peaks) else "基準値を超えるピークはありませんでした")

<small>※ ここでは単純な回帰分析を使っているので、上の（混合モデルなどで計算された）結果とは、ピークの高さや位置が少し違います。</small>

## 多数の遺伝子のeQTL解析（cis と trans）

　eQTL解析の本領は、**たくさんの遺伝子**についてまとめて解析することで、遺伝子の制御関係の全体像をつかめることです。

> ### 🗳 Slido Q1
> **多数の遺伝子のeQTL解析をおこなうと、cis-eQTL と trans-eQTL のどちらが多く見つかると思いますか？**
>
> - cis-eQTL
> - trans-eQTL
> - 同じくらい
>
> <small>下のコードセルを実行する前に回答してください。</small>

In [ ]:
n_genes = 2000   # 解析する遺伝子の数（多いほど時間がかかります。2000遺伝子で30秒ほど）

## 発現している遺伝子からランダムに選ぶ
expressed_eqtl = TMM_eQTL[TMM_eQTL.mean(axis=1) >= 5].index
genes = np.random.default_rng(0).choice(expressed_eqtl, n_genes, replace=False)
threshold = -np.log10(0.05 / X.shape[1])

## 250遺伝子ずつまとめて解析し、各遺伝子で最も強いピークを記録する
rows = []
for k in range(0, n_genes, 250):
    batch = genes[k:k + 250]
    s = eqtl_scan(np.log2(TMM_eQTL.loc[batch].values.T + 1))
    best = s.argmax(axis=0)
    for gene, i in zip(batch, best):
        rows.append({"gene": gene, "score": s[i, list(batch).index(gene)],
                     "eqtl_chr": markers["Chrom"][i], "eqtl_pos": markers["Position"][i]})
eqtl_table = pd.DataFrame(rows)
eqtl_table = eqtl_table[eqtl_table["score"] >= threshold].copy()
eqtl_table["gene_chr"] = gene_pos.loc[eqtl_table["gene"], "chr"].values
eqtl_table["gene_pos"] = gene_pos.loc[eqtl_table["gene"], "pos"].values
eqtl_table["type"] = [classify(g, c, p) for g, c, p in zip(eqtl_table["gene"], eqtl_table["eqtl_chr"], eqtl_table["eqtl_pos"])]
print(f"eQTLが見つかった遺伝子: {len(eqtl_table)} / {n_genes}")
print(eqtl_table["type"].value_counts())

　結果を、横軸にeQTLの位置、縦軸に遺伝子の位置をとった **eQTLマップ** に描いてみましょう。

In [ ]:
## 染色体をつなげた通し番号の位置（Mb）に変換する
chrom_len = markers.groupby("Chrom")["Position"].max()
offset = chrom_len.cumsum().shift(fill_value=0)
to_genome = lambda c, p: (offset[c] + p) / 1e6

plt.figure(figsize=(7, 7))
for t, color in [("cis", "tab:blue"), ("trans", "tab:red")]:
    sub = eqtl_table[eqtl_table["type"] == t]
    plt.scatter([to_genome(c, p) for c, p in zip(sub["eqtl_chr"], sub["eqtl_pos"])],
                [to_genome(c, p) for c, p in zip(sub["gene_chr"], sub["gene_pos"])], s=8, color=color, label=t)
for o in offset:
    plt.axvline(o / 1e6, color="gray", lw=0.3); plt.axhline(o / 1e6, color="gray", lw=0.3)
plt.xlabel("Position of eQTL (Mb, chr01 → chr12)"); plt.ylabel("Position of gene (Mb, chr01 → chr12)")
plt.legend(); plt.show()

## trans-eQTLが集まっている場所（2 Mbごと）
trans = eqtl_table[eqtl_table["type"] == "trans"]
print("trans-eQTLが多い領域（上位5）")
print(trans.groupby([trans["eqtl_chr"], (trans["eqtl_pos"] // 2_000_000 * 2).rename("Mb")]).size().sort_values(ascending=False).head(5))

#### 確認ポイント
- 見つかったeQTLの多くは cis-eQTL で、eQTLマップの**対角線上**に並びました。遺伝子自身のプロモーターなどの変異が、その遺伝子の発現量を変えている例が多いことを示しています。
- trans-eQTL（対角線から外れた点）の中に、同じ位置（縦の列）に集まっているものはありますか？ 多くの遺伝子のtrans-eQTLが集まる場所を **eQTLホットスポット** と呼びます。ホットスポットには、たくさんの遺伝子の発現をまとめて制御する遺伝子（転写因子など）がある可能性があります。

### 課題: 調べた結果をつなげて考える
　eQTLホットスポットが見つかったとします。その領域に、制御の中心となる遺伝子があることを確かめるには、どのような解析を組み合わせればよいでしょうか？ この授業で学んだ方法（共発現解析、モチーフの濃縮解析、DAP-seq など）を使って考えてみましょう。

# 4. 最近のRNA-seq技術

ここでは最近しばしば活用され始めている新たなRNA-seq技術について触れます。

 ## Single cell RNAseq (scRNA-seq)

 従来のRNA-seqでは、サンプル(組織)をすり潰してRNAを抽出していたので、細胞レベルでは区別されず、まとめてRNAを読んでいた形でした。

そのため、例えば病原菌に感染させた細胞でどの様な遺伝子が発現しているのかを見たい、といった場合にも、採集した組織の中には健全な細胞が混ざっている様な状態でした。

 しかし、近年、Single cell RNA-seqと呼ばれる、細胞レベルでRNA-seqを行うことが可能になってきています。

 参考: [10xGenomicsの説明(scRNAseq)](https://www.10xgenomics.com/blog/single-cell-rna-seq-an-introductory-overview-and-tools-for-getting-started)

 <img src="https://github.com/CropEvol/lecture/blob/master/textbook_2022/images/scRNAseq.png?raw=true" alt="scRNAseq" height="300px" align="middle">



### single cell RNA-seq のデータ解析を体験する

　single cell RNA-seq（scRNA-seq）では、数千〜数万個の細胞について、1細胞ごとに遺伝子の発現量が得られます。データは「細胞 × 遺伝子」の大きな表になり、次のような流れで解析します。

1. 細胞ごとに発現量を正規化する
1. 主成分分析（PCA）で、データの情報をまとめる
1. 発現パターンが似た細胞どうしをつなぎ、**クラスタリング**で細胞のグループに分ける
1. **UMAP** という方法で、細胞を2次元の図に配置する（似た細胞が近くに集まる）
1. 各クラスターで特に高く発現している遺伝子（**マーカー遺伝子**）を探し、どんな種類の細胞かを推定する

　ここでは、scRNA-seqの解析で最もよく使われるPythonのライブラリ **[Scanpy](https://scanpy.readthedocs.io/)** に付属している練習用データ（ヒトの血液の細胞700個、遺伝子765個、正規化・PCAまで済み）を使って、3〜5の流れを体験します。植物のデータではありませんが、解析の流れは植物でもまったく同じです。

In [ ]:
import scanpy as sc

adata = sc.datasets.pbmc68k_reduced()      # 練習用データ（細胞 × 遺伝子）
print(adata)

## 3. 似た細胞どうしをつなぎ、クラスタリング（Leiden法）する
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=30)
sc.tl.leiden(adata, resolution=0.8, flavor="igraph", n_iterations=2, key_added="cluster")
## 4. UMAPで2次元に配置する
sc.tl.umap(adata)
sc.pl.umap(adata, color="cluster", legend_loc="on data", title="Clusters")

　1つ1つの点が1つの細胞で、色がクラスター（細胞のグループ）です。次に、各クラスターのマーカー遺伝子を探します。

In [ ]:
## 5. 各クラスターで特に高く発現している遺伝子（マーカー遺伝子）を探す
sc.tl.rank_genes_groups(adata, "cluster", method="wilcoxon")
markers_table = pd.DataFrame(adata.uns["rank_genes_groups"]["names"]).head(5)
markers_table

### 練習4-1: 細胞の種類を推定する

　血液の細胞の種類ごとに、よく知られたマーカー遺伝子があります。

|細胞の種類|マーカー遺伝子の例|
|:--|:--|
|T細胞|CD3D, CD3E|
|B細胞|CD79A, MS4A1|
|NK細胞|NKG7, GNLY|
|単球|LYZ, S100A8, FCGR3A|
|樹状細胞|FCER1A, CST3|

　次のコードセルの `genes` に、表のマーカー遺伝子を入れて実行すると、UMAPの図上で、その遺伝子が発現している細胞が明るい色で表示されます。上のマーカー遺伝子の表と合わせて、各クラスターがどの細胞の種類かを推定してみましょう。

In [ ]:
genes = ["CD3D", "CD79A", "NKG7", "LYZ"]   # 発現を見たいマーカー遺伝子

sc.pl.umap(adata, color=genes, ncols=4, cmap="viridis")

　答え合わせとして、このデータにあらかじめ付けられている細胞の種類（`bulk_labels`）と比べてみましょう。

In [ ]:
sc.pl.umap(adata, color="bulk_labels", title="Cell types (given labels)")
pd.crosstab(adata.obs["cluster"], adata.obs["bulk_labels"])

### 従来のRNA-seqとの違い
　従来のRNA-seq（バルクRNA-seq）で得られる発現量は、組織の中の全細胞の**平均**です。もし、ある遺伝子がB細胞だけで強く発現していても、組織の中にB細胞が少なければ、平均ではほとんど目立ちません。scRNA-seqなら、どの種類の細胞で、どの遺伝子が発現しているのかを区別できます。

<small>※ 植物でも、シロイヌナズナの根などで、細胞の種類ごとの発現の地図（シングルセルアトラス）が作られています。植物の細胞は細胞壁に囲まれているので、酵素で細胞壁を溶かしてプロトプラストにするか、細胞の核を取り出して解析する（single nucleus RNA-seq）などの工夫が必要です。</small>

## Spatial Transcriptomics

また、このSingle cell RNA-seqを応用することで、細胞毎に発現量を測定するだけではなく、組織の中でどの部分の細胞がどれだけ発現しているのか、という空間的な情報を調べる様な技術も近年は発展してきています。

参考: [10xGenomicsの説明(Spatial Transcriptomics)](https://www.10xgenomics.com/spatial-transcriptomics)


## その他の新しい技術

- **ロングリードRNA-seq**（PacBio の Iso-Seq、Oxford Nanopore の direct RNA-seq など）: mRNAを全長で読めるので、選択的スプライシングで生じる転写産物（アイソフォーム）の違いを正確に調べられる。ナノポアでは、RNAの化学修飾も検出できる
- **深層学習による発現予測**: DNA配列だけから、遺伝子の発現量や転写因子の結合を予測するAIモデルが開発されている（例: Enformer, Avsec et al. 2021, *Nature Methods* 18: 1196–1203）。植物でも、ゲノム配列を学習した言語モデルの開発が進んでいる

ただし、まだまだこれらの技術は登場して間もない技術で、シーケンスの値段などが非常に高く、コストがかかる技術です。

 また、これらの技術を用いて何か革命的な発見が出来たのか、と言われると、まだまだそういった結果は出てきていないので、

 今後より普及が進み、どの様に扱うと良いか、という部分が洗練されていく必要があるなという印象です。

# まとめ

　今回、遺伝子ネットワーク解析の概要と解析の流れをみてきました。

ネットワーク解析を用いることで、どんな遺伝子が一緒に働いているのか、遺伝子の制御関係がどうなっているのか、といったことを調べることが出来ます。

- 共発現解析: 発現パターンの似た遺伝子群（モジュール）を見つける。サンプル数が少ないと、共通の時間変化なども共発現に見えてしまう
- 転写因子と結合モチーフ: 短いモチーフは偶然にもよく現れるので、注目する遺伝子群に濃縮しているかを統計的に調べる
- eQTL解析: 発現量を変える遺伝子座を探す。cis-eQTLが多く、trans-eQTLのホットスポットは制御の中心となる遺伝子の手がかりになる
- single cell RNA-seq: 細胞ごとの発現から、細胞の種類と、そこで働く遺伝子を調べられる

RNA-seqは実験結果の確認のために発現量を測定する目的で行うこともありますし、RNA-seqのデータから何か新たな発見のヒントを得るために行ったりと、幅広い目的で行われるシーケンス解析です。

皆さんの中でも、RNA-seqデータを使用する人は多いかと思うので、どのような事が出来るかを知っておくと役に立つかもしれません。

<div align="right"><a href="https://github.com/CropEvol/lecture#section2">実習表ページに戻る</a></div>